In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load MidiDatasets

In [3]:
from pathlib import Path
import shutil

# midi_directory and excluded_directory are assumed to be Path objects or strings
midi_directory = Path("/content/drive/MyDrive/capstone_team_3/MidiDatasets/590-Classical-music-midi")


In [4]:
# Load the MIDI files and extract features using pretty_midi
import pretty_midi
import os   
midi_files = []
for root, dirs, files in os.walk(midi_directory):
    for file in files:
        if file.endswith('.mid') or file.endswith('.midi'):
            midi_files.append(os.path.join(root, file))
print(f"Total MIDI files found: {len(midi_files)}")

Total MIDI files found: 292


In [5]:
#  extract features from MIDI data with duration, pitch_diff, velocity
import numpy as np
def extract_features(midi_data):
        features = []
        previous_pitch = None
        for instrument in midi_data.instruments:
            for note in instrument.notes:
                #  duration, pitch, velocity, pitch_diff    
                pitch_diff = note.pitch - previous_pitch if previous_pitch is not None else 0
                features.append([note.end - note.start, pitch_diff, note.velocity])
                previous_pitch = note.pitch
        return np.array(features)   

In [6]:
midi_features = []
for midi_path in midi_files:
    midi_data = pretty_midi.PrettyMIDI(midi_path)
    features = extract_features(midi_data)
    midi_features.append(features)
print(f"Extracted features from {len(midi_features)} MIDI files.")

Extracted features from 292 MIDI files.


In [11]:
# Convert the list of features into a format suitable for training (e.g., numpy array)
midi_features_array = np.array(midi_features, dtype=object)  # Using dtype=object to handle variable-length feature arrays
print(f"Shape of MIDI features array: {midi_features_array.shape}")
#Store the extracted features for later use as CSV or binary file
import pandas as pd
# Convert the list of features into a DataFrame for easier storage and manipulation
features_df = pd.DataFrame(midi_features_array, columns=['features'])
# Add file names or identifiers if needed (optional)
features_df['file_name'] = [os.path.basename(path) for path in midi_files]  
# Save the DataFrame to picklet format for efficient storage
features_df.to_pickle('/content/drive/MyDrive/capstone_team_3/midi_features.pkl')
print("MIDI features saved to pickle file.") 

Shape of MIDI features array: (292,)
MIDI features saved to pickle file.


In [12]:
#load the features from the pickle file
loaded_features_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/midi_features.pkl')
print("MIDI features loaded from pickle file.")
print(f"Shape of loaded MIDI features DataFrame: {loaded_features_df.shape}")

MIDI features loaded from pickle file.
Shape of loaded MIDI features DataFrame: (292, 2)


In [13]:
# Convert the pitch into  windowed sequences for training a sequence model
def create_sequences(features, sequence_length=256):
    sequences = []
    for i in range(len(features) - sequence_length):
        seq = features[i:i + sequence_length]
        sequences.append(seq)
    return np.array(sequences)
# use loaded_features_df to create sequences for training
# Covvert the all the midi features into sequences for training,  Each sequence will be a window of 50 notes (or other features) for training a model like an LSTM or Transformer. Label the sequences appropriately for supervised learning, give the name of the midi file as the label for each sequence.
all_sequences = []
for midi_path, features in zip(loaded_features_df['file_name'], loaded_features_df['features']):
    if features.size > 0:  # Check if there are features to avoid errors
        sequences = create_sequences(features)
        file_name = os.path.basename(midi_path)  # Get the file name from the path
        labels = [file_name] * len(sequences)  # Label each sequence with the MIDI file name
        all_sequences.extend(zip(sequences, labels))
print(f"Total sequences created: {len(all_sequences)}")

# Create a DF for the sequences and labels
sequences_df = pd.DataFrame(all_sequences, columns=['sequence', 'label'])
print(f"Shape of sequences DataFrame: {sequences_df.shape}")

Total sequences created: 630987
Shape of sequences DataFrame: (630987, 2)


In [14]:
sequences_df.head()

,sequence,label
0,"[[0.12396699999999994, 0.0, 80.0], [0.71005899...",muss_6.mid
1,"[[0.7100589999999998, 7.0, 88.0], [0.088757375...",muss_6.mid
2,"[[0.08875737500000014, -1.0, 80.0], [0.0887573...",muss_6.mid
3,"[[0.08875737500000014, 1.0, 80.0], [2.080926, ...",muss_6.mid
4,"[[2.080926, -1.0, 92.0], [0.24242433333333313,...",muss_6.mid


In [20]:
# Store the sequences and labels for training
sequences_df.to_pickle('/content/drive/MyDrive/capstone_team_3/data_set/classic_midi_sequences_256.pkl')
print("MIDI sequences saved to pickle file.")   

MIDI sequences saved to pickle file.


In [21]:
# Load the sequences and labels for training
loaded_sequences_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/data_set/classic_midi_sequences_256.pkl')
print("MIDI sequences loaded from pickle file.")
print(f"Shape of loaded sequences DataFrame: {loaded_sequences_df.shape}")  

MIDI sequences loaded from pickle file.
Shape of loaded sequences DataFrame: (630987, 2)


In [23]:
# Split the dataset into training and testing sets
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(loaded_sequences_df, test_size=0.2, random_state=42)
print(f"Training set size: {len(train_df)}")
print(f"Testing set size: {len(test_df)}")  

Training set size: 504789
Testing set size: 126198


In [24]:
test_sequences = []
test_midi_directory = Path("/content/drive/MyDrive/capstone_team_3/MidiDatasets/TestingSamples/MidiOutputs")
test_files=[]
for root, dirs, files in os.walk(test_midi_directory):
    for file in files:
        if file.endswith('.mid') or file.endswith('.midi'):
            midi_path = os.path.join(root, file)
            midi_data = pretty_midi.PrettyMIDI(midi_path)
            features = extract_features(midi_data)
            file_name = os.path.basename(midi_path)  # Get the file name from the path
            test_sequences.append((features, file))  # Store features and filename as label
print(f"Total test MIDI sequences created: {len(test_sequences)}")



/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Total test MIDI sequences created: 50


In [25]:
# merge the test sequences ,filename and features into a single list for evaluation
merged_test_sequences = []
# Store each sequence with the MIDI file name as label
for features, filename in test_sequences:
    if features.size > 0:
        sequences = create_sequences(features)
        for seq in sequences:
            merged_test_sequences.append((seq, filename))  # Store each sequence with the MIDI file name as label

In [27]:
# Create a DATAFRAME for the merged test sequences
external_test_df = pd.DataFrame(merged_test_sequences, columns=['sequence', 'label'])
print(f"Shape of merged test sequences DataFrame: {external_test_df.shape}")
print("Sample merged test sequences:")
external_test_df.head()

Shape of merged test sequences DataFrame: (203723, 2)
Sample merged test sequences:


,sequence,label
0,"[[0.7999999999999998, 0.0, 44.0], [0.300000000...",Piano Sonata n12 K332.mid
1,"[[0.30000000000000027, 4.0, 47.0], [0.79999999...",Piano Sonata n12 K332.mid
2,"[[0.7999999999999998, 3.0, 49.0], [0.304166666...",Piano Sonata n12 K332.mid
3,"[[0.3041666666666667, -3.0, 52.0], [0.79999999...",Piano Sonata n12 K332.mid
4,"[[0.7999999999999998, 1.0, 54.0], [0.299999999...",Piano Sonata n12 K332.mid


In [29]:
# Save the sequences DataFrame to a pickle file for later use
external_test_df.to_pickle('/content/drive/MyDrive/capstone_team_3/data_set/external_test_sequences.pkl')
print("External test sequences saved to pickle file.")  

External test sequences saved to pickle file.
